# Plot Spice
This is a prototyping notebook for plotting SPICE results locally using our plot framework. This is only intended for prototyping and testing purposes and should be deleted before final submission.

In [76]:
from pathlib import Path
import sys
from analysis_util import find_switching_time, get_sample, apply_time_shift

from bokeh.io import output_notebook
output_notebook()

from plot_framework import iplot, ioverlay, isweep, istack, isweep_overlay
from ngspice_loader import load_wrdata, load_sweep

PROJECT_ROOT = Path("/Users/phevos/Cac_characterization_PD_4.14.2026/CaC_Spring26")
SPICE_DIR = PROJECT_ROOT / "spice"

Loading BokehJS ...

## Experiment Block

In [77]:
# Single-file waveform viewer
# traces = load_wrdata(
#     SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",
#     ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
# )

# istack(
#     layers=[
#         {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
#         {"RST": traces["RST"]},
#         {"UP": traces["UP"], "DOWN": traces["DOWN"]},
#     ],
#     title="FF1 phase detector — clk_in leads",
#     xlabel="Time (ns)",
#     ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
# )

# print(traces)

# Plot one signal
# iplot(*traces["DOWN"], title="UP output", xlabel="Time (ns)", ylabel="V")

# Overlay all signals
# ioverlay(traces, title="FF1 — clk_in leads", xlabel="Time (ns)", ylabel="Voltage (V)")

# Overlay a subset
# ioverlay(
#     {k: traces[k] for k in ["CLK_IN", "UP", "DOWN"]},
#     title="FF1 — clock vs outputs",
#     xlabel="Time (ns)", ylabel="V",
# )

# Compare across test phases (sweep)
# sweep = load_sweep(
#     {
#         "clk_in leads":  (SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",  ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"]),
#         "clk_out leads": (SPICE_DIR / "results" / "pd_syn_ff1_clkout_lead_results.csv", ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"]),
#     },
#     signal="DOWN",
# )
# # {"clk_in leads": (t, v), "clk_out leads": (t, v), "near zero": (t, v)}
# print(sweep["clk_in leads"])
# print(sweep["clk_out leads"])

# iplot(*sweep["clk_in leads"], title="UP output", xlabel="Time (ns)", ylabel="V")

# isweep(sweep, title="DOWN across phases", xlabel="Time (ns)", ylabel="V")

# # Compare across detectors (sweep + overlay)
# groups = load_sweep_overlay(
#     sweep_axis={
#         "clk_in leads":  {
#             "FF1": ("results/ff1/clk_in_leads_results.csv",  ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#             "PFD": ("results/pfd/clk_in_leads_results.csv",  ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#         },
#         "clk_out leads": {
#             "FF1": ("results/ff1/clk_out_leads_results.csv", ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#             "PFD": ("results/pfd/clk_out_leads_results.csv", ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#         },
#     },
#     signal="DOWN",
# )
# isweep_overlay(groups, title="DOWN: FF1 vs PFD", xlabel="Time (ns)", ylabel="V")

## NAND DCDL

In [78]:
# v(A) v(Y) v(Q0_node) v(Q1_node) v(Q2_node) v(Q3_node)
traces = load_wrdata(
    SPICE_DIR / "tmp_results" / "nand_dcdl_results.csv",
    ["A", "Y", "Q0_node", "Q1_node", "Q2_node", "Q3_node"],
)

istack(
    layers=[
        {"Q0": traces["Q0_node"], "Q1": traces["Q1_node"],
         "Q2": traces["Q2_node"], "Q3": traces["Q3_node"]},
        {"A": traces["A"]},
        {"Y": traces["Y"]},
    ],
    title="NAND DCDL",
    xlabel="Time (ns)",
    ylabels=["Q[0:3] (V)", "A (V)", "Y (V)"],
)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/phevos/Cac_characterization_PD_4.14.2026/CaC_Spring26/spice/tmp_results/nand_dcdl_results.csv'

In [ ]:
# Find propagation delay
for time_interval in [(0, 25), (25, 50), (50, 75), (75, 100)]:
    t_start, t_end = time_interval
    A_switch_time = find_switching_time(traces["A"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    Y_switch_time = find_switching_time(traces["Y"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    prop_delay_LH = Y_switch_time - A_switch_time
    print(f"Low to High Propagation delay from t={t_start} to t={t_end}: {prop_delay_LH : .5f} ns")

KeyError: 'A'

In [ ]:
delta_t = 1
sampled_traces = {}
for q_val, time_interval in ({"Q0": (0, 25), "Q1": (25, 50), "Q2": (50, 75), "Q3": (75, 100)}).items():
    t_start, t_end = time_interval
    A_switch_time = find_switching_time(traces["A"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    Y_switch_time = find_switching_time(traces["Y"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    sampled_trace = get_sample(traces["Y"], t_start=Y_switch_time - delta_t, t_end = Y_switch_time + delta_t)
    sampled_traces[q_val] = apply_time_shift(sampled_trace, time_shift=-A_switch_time)

ioverlay(
    {k: sampled_traces[k] for k in ["Q0", "Q1", "Q2", "Q3"]},
    title="Rising edge of ",
    xlabel="Propagation delay time relative to switch time of A (ns)", ylabel="Y (V)",
)

KeyError: 'A'

## Controller 2 Mode

In [ ]:
from analysis_util import decode_ctrl

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_2mode.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller — Post-Layout SPICE",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

## Controller Variable Step

In [ ]:
from analysis_util import decode_ctrl

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_variable_step.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller — Post-Layout SPICE",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

## Controller Saturate

In [ ]:
from analysis_util import decode_ctrl

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_saturate.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller — Post-Layout SPICE",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

## FF1 Phase Detector

In [ ]:
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [ ]:
import importlib
import analysis_util
importlib.reload(analysis_util)

from analysis_util import find_switching_times
from ngspice_loader import load_wrdata

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

t_start, t_end = 19.5, 21.0

clkin_rises = find_switching_times(
    traces["CLK_IN"], t_start=t_start, t_end=t_end, edge="rising"
)
up_rises = find_switching_times(
    traces["UP"], t_start=t_start, t_end=t_end, edge="rising"
)

print("CLK_IN window edges:", clkin_rises)
print("UP window edges:", up_rises)

if not clkin_rises or not up_rises:
    print(f"t={t_start} to t={t_end}: missing edge")
else:
    prop_delay = up_rises[0] - clkin_rises[0]
    print(f"CLK_IN -> UP delay from t={t_start} to t={t_end}: {prop_delay:.5f} ns")


CLK_IN window edges: [np.float64(20.025)]
UP window edges: [np.float64(20.37377924125149)]
CLK_IN -> UP delay from t=19.5 to t=21.0: 0.34878 ns


In [ ]:
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [ ]:
import importlib
import analysis_util
importlib.reload(analysis_util)

from analysis_util import find_switching_times
from ngspice_loader import load_wrdata

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

t_start, t_end = 19.5, 23.0

clkout_rises = find_switching_times(
    traces["CLK_OUT"], t_start=t_start, t_end=t_end, edge="rising"
)
down_rises = find_switching_times(
    traces["DOWN"], t_start=t_start, t_end=t_end, edge="rising"
)

print("CLK_OUT window edges:", clkout_rises)
print("DOWN window edges:", down_rises)

if not clkout_rises or not down_rises:
    print(f"t={t_start} to t={t_end}: missing edge")
else:
    prop_delay = down_rises[0] - clkout_rises[0]
    print(f"CLK_OUT -> DOWN delay from t={t_start} to t={t_end}: {prop_delay:.5f} ns")


CLK_OUT window edges: [np.float64(20.025)]
DOWN window edges: [np.float64(22.373245009749397)]
CLK_OUT -> DOWN delay from t=19.5 to t=23.0: 2.34825 ns


## Edge Phase Detector

In [ ]:
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="Edge phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [ ]:
import importlib
import analysis_util
importlib.reload(analysis_util)

from analysis_util import find_switching_times
from ngspice_loader import load_wrdata

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

t_start, t_end = 19.5, 21.0

clkin_rises = find_switching_times(
    traces["CLK_IN"], t_start=t_start, t_end=t_end, edge="rising"
)
up_rises = find_switching_times(
    traces["UP"], t_start=t_start, t_end=t_end, edge="rising"
)

print("CLK_IN window edges:", clkin_rises)
print("UP window edges:", up_rises)

if not clkin_rises or not up_rises:
    print(f"t={t_start} to t={t_end}: missing edge")
else:
    clkin_switch_time = clkin_rises[0]
    up_switch_time = up_rises[0]
    prop_delay = up_switch_time - clkin_switch_time
    print(f"CLK_IN -> UP delay from t={t_start} to t={t_end}: {prop_delay:.5f} ns")


CLK_IN window edges: [np.float64(20.025)]
UP window edges: [np.float64(20.267811070249703)]
CLK_IN -> UP delay from t=19.5 to t=21.0: 0.24281 ns


In [ ]:
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="Edge phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [ ]:
import importlib
import analysis_util
importlib.reload(analysis_util)

from analysis_util import find_switching_times
from ngspice_loader import load_wrdata

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

t_start, t_end = 19.5, 21.0
clkout_rises = find_switching_times(traces["CLK_OUT"], t_start, t_end, edge="rising")
down_rises = find_switching_times(traces["DOWN"], t_start, t_end, edge="rising")

prop_delay = down_rises[0] - clkout_rises[0]
print(f"CLK_OUT -> DOWN delay: {prop_delay:.5f} ns")


CLK_OUT -> DOWN delay: 0.24281 ns


Phase 

## PFD Phase Detector

In [81]:
traces = load_wrdata(
    SPICE_DIR / "results" / "phase_detector_syn_pfd_clkin_leads.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="pfd phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [80]:
import importlib
import analysis_util
importlib.reload(analysis_util)

from analysis_util import find_switching_times
from ngspice_loader import load_wrdata

traces = load_wrdata(
    SPICE_DIR / "results" / "phase_detector_syn_pfd_clkin_leads.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

for t_start, t_end in [(28, 32), (58, 62), (88, 92)]:
    clkin_rises = find_switching_times(
        traces["CLK_IN"], t_start=t_start, t_end=t_end, edge="rising"
    )
    up_rises = find_switching_times(
        traces["UP"], t_start=t_start, t_end=t_end, edge="rising"
    )

    print("CLK_IN window edges:", clkin_rises)
    print("UP window edges:", up_rises)

    if not clkin_rises or not up_rises:
        print(f"t={t_start} to t={t_end}: missing edge")
        continue

    clkin_switch_time = clkin_rises[0]
    up_switch_time = up_rises[0]

    prop_delay = up_switch_time - clkin_switch_time
    print(f"CLK_IN -> UP delay from t={t_start} to t={t_end}: {prop_delay:.5f} ns")


CLK_IN window edges: [np.float64(30.025)]
UP window edges: [np.float64(30.379156345137122)]
CLK_IN -> UP delay from t=28 to t=32: 0.35416 ns
CLK_IN window edges: [np.float64(60.025)]
UP window edges: [np.float64(60.378951574918695)]
CLK_IN -> UP delay from t=58 to t=62: 0.35395 ns
CLK_IN window edges: [np.float64(90.025)]
UP window edges: [np.float64(90.3789515749187)]
CLK_IN -> UP delay from t=88 to t=92: 0.35395 ns


In [83]:
traces = load_wrdata(
    SPICE_DIR / "results" / "phase_detector_syn_pfd_clkout_leads.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="pfd phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [84]:
import importlib
import analysis_util
importlib.reload(analysis_util)

from analysis_util import find_switching_times
from ngspice_loader import load_wrdata

traces = load_wrdata(
    SPICE_DIR / "results" / "phase_detector_syn_pfd_clkout_leads.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

for t_start, t_end in [(28, 32), (58, 62), (88, 92)]:
    clkout_rises = find_switching_times(
        traces["CLK_OUT"], t_start=t_start, t_end=t_end, edge="rising"
    )
    down_rises = find_switching_times(
        traces["DOWN"], t_start=t_start, t_end=t_end, edge="rising"
    )

    print("CLK_OUT window edges:", clkout_rises)
    print("DOWN window edges:", down_rises)

    if not clkout_rises or not down_rises:
        print(f"t={t_start} to t={t_end}: missing edge")
        continue

    clkout_switch_time = max((t for t in clkout_rises if t <= down_rises[0]), default=None)
    down_switch_time = down_rises[0]

    if clkout_switch_time is None:
        print(f"t={t_start} to t={t_end}: no matching CLK_OUT edge")
        continue

    prop_delay = down_switch_time - clkout_switch_time
    print(f"CLK_OUT -> DOWN delay from t={t_start} to t={t_end}: {prop_delay:.5f} ns")


CLK_OUT window edges: [np.float64(30.025)]
DOWN window edges: [np.float64(30.378207075755498)]
CLK_OUT -> DOWN delay from t=28 to t=32: 0.35321 ns
CLK_OUT window edges: [np.float64(60.025)]
DOWN window edges: [np.float64(60.37799399810875)]
CLK_OUT -> DOWN delay from t=58 to t=62: 0.35299 ns
CLK_OUT window edges: [np.float64(90.025)]
DOWN window edges: [np.float64(90.37799399810875)]
CLK_OUT -> DOWN delay from t=88 to t=92: 0.35299 ns


## xor1 Phase Detector

In [ ]:
traces = load_wrdata(
    SPICE_DIR / "results" / "phase_detector_syn_xor1_clkin_leads.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="xor1 phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)